In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg , sum , year, month, dayofmonth , lit
from sqlalchemy import create_engine
import sqlite3
import pandas as pd

# Para crear la sessión de spark
spark = SparkSession.builder \
    .appName("Proyecto Failed Logins") \
    .getOrCreate()
print(spark)
print("Versión de Spark:", spark.version)


Versión de Spark: 3.5.0


In [19]:
# Cargar dataset limpio
df_spark = spark.read.csv("Data/dataset_clean_pandas.csv", header=True, inferSchema=True)

In [20]:
# Mostrar el schema y las 5 primeras filas
df_spark.printSchema()
df_spark.show(5)


root
 |-- event_id: integer (nullable = true)
 |-- username: string (nullable = true)
 |-- role: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- location: string (nullable = true)
 |-- device: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- login_attempts: string (nullable = true)
 |-- source_port: string (nullable = true)
 |-- last_success_login: string (nullable = true)
 |-- alert_flag: integer (nullable = true)

+--------+--------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-------------------+----------+
|event_id|username| role|status|          timestamp|    ip_address|location| device|account_status|login_attempts|source_port| last_success_login|alert_flag|
+--------+--------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-----

In [33]:
# Selecciono las columnas mas relevantes para un acceso de un usuario
df_select = df_spark.select(
    "event_id", "username", "status","location","source_port", "alert_flag"
)
df_select.show(10)

+--------+--------+-------+--------+-----------+----------+
|event_id|username| status|location|source_port|alert_flag|
+--------+--------+-------+--------+-----------+----------+
|       1|    root| failed|      de|        443|         0|
|       2|    jdoe| failed|   spain|        443|         1|
|       3| service| failed|  españa|       3389|         0|
|       4|   admin| failed|   spain|    unknown|         0|
|       5|  backup| failed|      es|       8080|         0|
|       6| service| failed|     usa|         22|         1|
|       7|   guest|success|      es|        443|         0|
|       8|  backup|success|     usa|       3389|         1|
|       9|    jdoe| locked|      es|       8080|         1|
|      10|operator| failed|  españa|       3389|         0|
+--------+--------+-------+--------+-----------+----------+
only showing top 10 rows



In [18]:
# Filtrar solo los eventos con status failed y puertos válidos.
df_failed = df_spark.filter(
    (col("status") == "failed") & (col("source_port") != "unkown")
)
df_failed.show(10)

+--------+----------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-------------------+----------+
|event_id|  username| role|status|          timestamp|    ip_address|location| device|account_status|login_attempts|source_port| last_success_login|alert_flag|
+--------+----------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-------------------+----------+
|       1|      root| user|failed|2024-08-06 16:50:00|165.93.148.165| Germany|Unknown|      inactive|       Unknown|        443|2024-11-06 16:50:00|         0|
|       2|      jdoe|guest|failed|2024-06-19 22:06:00| 221.49.14.175|   Spain| Ubuntu|        locked|             5|        443|2024-06-06 22:06:00|         1|
|       3|   service| user|failed|2024-04-08 22:14:00| 242.73.33.181|   Spain|Windows|        active|             3|       3389|2024-12-06 22:14:00|         0|
|       5|    backup|admin|failed|2024-0

In [34]:
# Contar cuántos intentos ha hecho cada usuario con groupby
df_attempts = df_spark.groupBy("username").agg(
    count("event_id").alias("total_events"),
    count(when(col("status") == "failed", True)).alias("failed_events")
)
df_attempts.show(10)

+----------+------------+-------------+
|  username|total_events|failed_events|
+----------+------------+-------------+
|      root|        1020|          616|
|      jdoe|         965|          587|
|     admin|        1052|          608|
|    backup|         990|          597|
|it_manager|         972|          598|
|     guest|        1034|          611|
|    asmith|        1016|          619|
|   service|         972|          595|
|  sysadmin|         988|          576|
|  operator|         991|          594|
+----------+------------+-------------+



In [35]:
# Uso de join
df_users = df_spark.select("username", "location","device")

df_stats = df_spark.groupBy("username") \
                   .agg(count("*").alias("total_events"))
df_join = df_users.join(df_stats, on="username", how="inner")
df_join.show()

+----------+--------+-------+------------+
|  username|location| device|total_events|
+----------+--------+-------+------------+
|      root|      de|Unknown|        1020|
|      jdoe|   spain| Ubuntu|         965|
|   service|  españa|Windows|         972|
|     admin|   spain|Windows|        1052|
|    backup|      es|Windows|         990|
|   service|     usa|Unknown|         972|
|     guest|      es|Windows|        1034|
|    backup|     usa|Android|         990|
|      jdoe|      es|  Linux|         965|
|  operator|  españa|Android|         991|
|  sysadmin|      us|Windows|         988|
|  operator| germany|  Linux|         991|
|      jdoe| unknown| Ubuntu|         965|
|      jdoe|      fr|  Macos|         965|
|  sysadmin|   spain|  Macos|         988|
|it_manager|     usa|  Linux|         972|
|    backup|      es|  Macos|         990|
|  operator|  france|Windows|         991|
|      root|      de|Unknown|        1020|
|  sysadmin|      de| Ubuntu|         988|
+----------

In [ ]:
# Fase 4

In [ ]:
# Cerrar sparkSession
spark.stop()


In [37]:
# Para filtrar por status en este caso failed
df_spark = df_spark.filter(col("status").isin("failed"))
df_spark.show(10)


+--------+--------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-------------------+----------+
|event_id|username| role|status|          timestamp|    ip_address|location| device|account_status|login_attempts|source_port| last_success_login|alert_flag|
+--------+--------+-----+------+-------------------+--------------+--------+-------+--------------+--------------+-----------+-------------------+----------+
|       1|    root| user|failed|2024-08-06 16:50:00|165.93.148.165|      de|Unknown|      inactive|       Unknown|        443|2024-06-11 16:50:00|         0|
|       2|    jdoe|guest|failed|2024-06-19 22:06:00| 221.49.14.175|   spain| Ubuntu|        locked|       Unknown|        443|2024-06-06 22:06:00|         1|
|       3| service| user|failed|2024-08-04 22:14:00| 242.73.33.181|  españa|Windows|        active|           3.0|       3389|2024-12-06 22:14:00|         0|
|       4|   admin|guest|failed|2024-06-18 07:47:00|

In [38]:
# Agrupar por username, total_events, failed_events y success_events
df_stats = df_spark.groupBy("username").agg(
    count("*").alias("total_events"),  # total de eventos
    sum(when(col("status") == "failed", 1).otherwise(0)).alias("failed_events"),  # fallidos
    sum(when(col("status") == "success", 1).otherwise(0)).alias("success_events")  # exitosos
)

df_stats.show(10)

+----------+------------+-------------+--------------+
|  username|total_events|failed_events|success_events|
+----------+------------+-------------+--------------+
|      root|         616|          616|             0|
|      jdoe|         587|          587|             0|
|     admin|         608|          608|             0|
|    backup|         597|          597|             0|
|it_manager|         598|          598|             0|
|     guest|         611|          611|             0|
|    asmith|         619|          619|             0|
|   service|         595|          595|             0|
|  sysadmin|         576|          576|             0|
|  operator|         594|          594|             0|
+----------+------------+-------------+--------------+



In [5]:
# Dimensión usuarios
dim_users = df_spark.select("username", "role", "location", "device").dropDuplicates() \
    .withColumn("user_id", lit(None))  # Creamos columna de IDs vacía por ahora



In [6]:
# Dimensión puertos
dim_ports = df_spark.select("source_port").dropDuplicates() \
    .withColumn("port_id", lit(None))  # ID vacío por ahora


In [7]:

# Dimensión fechas
dim_dates = df_spark.select("timestamp").dropDuplicates() \
    .withColumn("timestamp", col("timestamp").cast("timestamp")) \
    .withColumn("date_id", lit(None)) \
    .withColumn("year", year("timestamp")) \
    .withColumn("month", month("timestamp")) \
    .withColumn("day", dayofmonth("timestamp"))


In [8]:

# Tabla de hechos
# Primero agregamos counts por usuario 
df_stats = df_spark.groupBy("username") \
    .agg(count("*").alias("total_events"))

# Join para generar tabla de hechos
fact_logins = df_spark.join(dim_users, on=["username", "role", "location", "device"], how="left") \
    .join(dim_ports, on="source_port", how="left") \
    .join(dim_dates, on="timestamp", how="left")

# Seleccionamos columnas de la tabla de hechos
fact_logins = fact_logins.select("user_id", "port_id", "date_id", "status", "login_attempts", "alert_flag")


In [11]:
# Convertimos a Pandas para escribir en SQLite
dim_users_pd = dim_users.toPandas()
dim_ports_pd = dim_ports.toPandas()
dim_dates_pd = dim_dates.toPandas()
fact_logins_pd = fact_logins.toPandas()

# Crear conexión con SQLite
engine = create_engine('sqlite:///warehouse_pyspark.db')

# Guardar dimensiones y tabla de hechos
dim_users_pd.to_sql("dim_users", engine, if_exists="replace", index=False)
dim_ports_pd.to_sql("dim_ports", engine, if_exists="replace", index=False)
dim_dates_pd.to_sql("dim_dates", engine, if_exists="replace", index=False)
fact_logins_pd.to_sql("fact_logins", engine, if_exists="replace", index=False)

print(" Guardado en warehouse_pyspark.db")

 Guardado en warehouse_pyspark.db


In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)

In [39]:
# Conectar a la base de datos
conn = sqlite3.connect('warehouse/warehouse_pyspark1.db')

# Consultar las tablas
print("Tablas en SQLite:")
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

# Mostrar los primeros 5 registros de cada tabla
print("\nDim Users:")
print(pd.read_sql_query("SELECT * FROM dim_users LIMIT 5;", conn))

print("\nDim Ports:")
print(pd.read_sql_query("SELECT * FROM dim_ports LIMIT 5;", conn))

print("\nDim Dates:")
print(pd.read_sql_query("SELECT * FROM dim_dates LIMIT 5;", conn))

print("\nFact Logins:")
print(pd.read_sql_query("SELECT * FROM fact_logins LIMIT 5;", conn))

# Cerrar la conexión
conn.close()


Tablas en SQLite:
          name
0    dim_users
1    dim_ports
2    dim_dates
3  fact_logins

Dim Users:
  username   role location   device  user_id
0    admin  admin      USA  Unknown        1
1    admin  admin   France  Unknown        2
2    admin  guest    Spain      Ios        3
3    admin   user  Unknown  Android        4
4    admin  guest      USA    Linux        5

Dim Ports:
  source_port  port_id
0          22        1
1        3389        2
2         443        3
3          80        4
4        8080        5

Dim Dates:
                    timestamp  date_id  year  month  day
0  2024-01-01 00:39:00.000000        1  2024      1    1
1  2024-01-01 00:42:00.000000        2  2024      1    1
2  2024-01-01 01:00:00.000000        3  2024      1    1
3  2024-01-01 01:30:00.000000        4  2024      1    1
4  2024-01-01 02:12:00.000000        5  2024      1    1

Fact Logins:
   user_id  port_id  date_id  status login_attempts  alert_flag
0      772        3     6984  failed       

In [14]:

# Crear conexión
conn = sqlite3.connect('warehouse/warehouse_pandas.db')

# Ver los primeros registros
query = """
SELECT * 
FROM fact_logins 
LIMIT 10;
"""

df_query = pd.read_sql(query, conn)
df_query


,user_id,port_id,date_id,status,login_attempts,alert_flag,fact_id
0,1,1,1,failed,Unknown,0,1
1,2,1,2,failed,5,1,2
2,3,2,3,failed,3,0,3
3,4,3,4,failed,4,0,4
4,5,4,5,failed,5,0,5
5,6,5,6,failed,6,1,6
6,7,1,7,success,2,0,7
7,8,2,8,success,3,1,8
8,9,4,9,locked,5,1,9
9,10,2,10,failed,4,0,10


In [15]:
# # Intentos fallidos de sesion por usuario
query = """
SELECT 
    u.username,
    COUNT(*) as total_intentos,
    SUM(CASE WHEN f.status = 'failed' THEN 1 ELSE 0 END) as fallos
FROM fact_logins f
JOIN dim_users u ON f.user_id = u.user_id
GROUP BY u.username
ORDER BY fallos DESC
LIMIT 10;
"""
df_query = pd.read_sql(query, conn)
df_query

,username,total_intentos,fallos
0,asmith,1016,619
1,root,1020,616
2,guest,1034,611
3,admin,1052,608
4,it_manager,972,598
5,backup,990,597
6,service,972,595
7,operator,991,594
8,jdoe,965,587
9,sysadmin,988,576


In [16]:
# ubicacion de los ataques

query = """
SELECT 
    u.location,
    COUNT(*) as total_alertas,
    COUNT(DISTINCT u.username) as usuarios_unicos
FROM fact_logins f
JOIN dim_users u ON f.user_id = u.user_id
WHERE f.alert_flag = 1
GROUP BY u.location
ORDER BY total_alertas DESC;
"""
df_query = pd.read_sql(query, conn)
df_query

,location,total_alertas,usuarios_unicos
0,USA,1072,10
1,Spain,1070,10
2,France,726,10
3,Germany,703,10
4,Unknown,374,10
